In [7]:

%pip install pandas rich ipywidgets



   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------------------------------- ------ 1.8/2.2 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.8 MB/s eta 0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
print(df['cosine_weighted'].iloc[0])
print(type(df['cosine_weighted'].iloc[0]))


[0.34644711940374345, 0.0, 0.0]
<class 'str'>


In [4]:
%pip install ipywidgets

import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

class ChunkRelabeler:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.original_df = self.df.copy()
        self.topic_names = sorted(set(t for sub in self.df['topic_assigned'].dropna().astype(str).apply(eval) for t in sub))
        self.cursor = 0
        self.sort_key = 'cosine_weighted' if 'cosine_weighted' in self.df.columns else 'cosine'
        self.df = self.df.sort_values(by=self.sort_key, ascending=False).reset_index(drop=True)
        self.build_ui()
    
    def build_ui(self):
        self.out = widgets.Output()
        self.label_box = widgets.SelectMultiple(
            options=self.topic_names,
            description="Labels",
            layout=widgets.Layout(width='auto', height='120px')
        )
        self.cosine_slider = widgets.FloatSlider(
            value=1.0,
            min=0.0,
            max=1.0,
            step=0.01,
            description='Manual Cosine:'
        )
        self.approve_button = widgets.Button(description="✅ Approve / Relabel", button_style='success')
        self.remove_label_button = widgets.Button(description="❌ Remove Labels", button_style='warning')
        self.remove_row_button = widgets.Button(description="🗑️ Remove Row", button_style='danger')
        self.next_button = widgets.Button(description="➡️ Next", button_style='info')
        self.prev_button = widgets.Button(description="⬅️ Prev", button_style='info')
        self.save_button = widgets.Button(description="💾 Save to CSV", button_style='primary')
        
        self.approve_button.on_click(self.approve_labels)
        self.remove_label_button.on_click(self.remove_labels)
        self.remove_row_button.on_click(self.remove_row)
        self.next_button.on_click(self.next_row)
        self.prev_button.on_click(self.prev_row)
        self.save_button.on_click(self.save_to_csv)

        control_box = widgets.VBox([
            widgets.HBox([self.prev_button, self.next_button, self.save_button]),
            self.label_box,
            self.cosine_slider,
            widgets.HBox([self.approve_button, self.remove_label_button, self.remove_row_button]),
            self.out
        ])
        display(control_box)
        self.show_current_row()

    def show_current_row(self):
        self.out.clear_output()
        row = self.df.iloc[self.cursor]
        with self.out:
            print(f"\nRow {self.cursor+1} / {len(self.df)} | Chunk ID: {row['filename']} ({row['year']})")
            print(f"Document Type: {row['document_type']} | Chunk #{row['chunk_num']}")
            print(f"Current Topics: {row['topic']}")
            print(f"Cosine: {row['cosine']} | Weighted: {row.get('cosine_weighted', 'n/a')}")
            print("\nText:\n" + "-"*80 + f"\n{row['chunk_text']}\n" + "-"*80)

    def approve_labels(self, _):
        labels = list(self.label_box.value)
        self.df.at[self.cursor, 'topic_assigned'] = str(labels)
        self.df.at[self.cursor, 'cosine'] = self.cosine_slider.value
        self.df.at[self.cursor, 'cosine_weighted'] = self.cosine_slider.value
        self.next_row(None)

    def remove_labels(self, _):
        self.df.at[self.cursor, 'topic_assigned'] = str([])
        self.df.at[self.cursor, 'cosine'] = 0.0
        self.df.at[self.cursor, 'cosine_weighted'] = 0.0
        self.next_row(None)

    def remove_row(self, _):
        self.df = self.df.drop(self.df.index[self.cursor]).reset_index(drop=True)
        if self.cursor >= len(self.df):
            self.cursor = len(self.df) - 1
        self.show_current_row()

    def next_row(self, _):
        self.cursor = (self.cursor + 1) % len(self.df)
        self.show_current_row()

    def prev_row(self, _):
        self.cursor = (self.cursor - 1) % len(self.df)
        self.show_current_row()

    def save_to_csv(self, _):
        self.df.to_csv("relabelled_output.csv", index=False)
        print("✅ Saved to relabelled_output.csv")

# Usage:
# relabeler = ChunkRelabeler("chunk_topics_corex_assignments.csv")


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import ast
from IPython.display import display, clear_output
import ipywidgets as widgets

class ChunkRelabeler:
    def __init__(self, csv_path):
        def safe_eval(val):
            if pd.isna(val):
                return []
            try:
                return ast.literal_eval(val)
            except Exception:
                return []

        self.df = pd.read_csv(csv_path)
        self.df['topic'] = self.df['topic'].apply(safe_eval)
        self.df['topic_assigned'] = self.df['topic_assigned'].apply(safe_eval)

        # Get topic names from any valid row
        self.topic_names = self._extract_topic_names()
        self.num_topics = len(self.topic_names)
        self.cursor = 0
        self.sort_key = 'cosine_weighted' if 'cosine_weighted' in self.df.columns else 'cosine'
        self.df = self.df.sort_values(by=self.sort_key, ascending=False).reset_index(drop=True)

        self.build_ui()

    def _extract_topic_names(self):
        for topics in self.df['topic']:
            if isinstance(topics, list) and all(isinstance(t, str) for t in topics):
                return topics
        raise ValueError("No valid topic name list found in the 'topic' column.")

    def build_ui(self):
        self.out = widgets.Output()
        self.label_box = widgets.SelectMultiple(
            options=self.topic_names,
            description="Labels",
            layout=widgets.Layout(width='auto', height='120px')
        )
        self.cosine_slider = widgets.FloatSlider(
            value=1.0, min=0.0, max=1.0, step=0.01,
            description='Manual Cosine:'
        )
        self.approve_button = widgets.Button(description="✅ Approve / Relabel", button_style='success')
        self.remove_label_button = widgets.Button(description="❌ Remove Labels", button_style='warning')
        self.remove_row_button = widgets.Button(description="🗑️ Remove Row", button_style='danger')
        self.next_button = widgets.Button(description="➡️ Next", button_style='info')
        self.prev_button = widgets.Button(description="⬅️ Prev", button_style='info')
        self.save_button = widgets.Button(description="💾 Save to CSV", button_style='primary')

        self.approve_button.on_click(self.approve_labels)
        self.remove_label_button.on_click(self.remove_labels)
        self.remove_row_button.on_click(self.remove_row)
        self.next_button.on_click(self.next_row)
        self.prev_button.on_click(self.prev_row)
        self.save_button.on_click(self.save_to_csv)

        control_box = widgets.VBox([
            widgets.HBox([self.prev_button, self.next_button, self.save_button]),
            self.label_box,
            self.cosine_slider,
            widgets.HBox([self.approve_button, self.remove_label_button, self.remove_row_button]),
            self.out
        ])
        display(control_box)
        self.show_current_row()

    def show_current_row(self):
        self.out.clear_output()
        row = self.df.iloc[self.cursor]
        assigned_binary = row['topic_assigned']
        assigned_names = [self.topic_names[i] for i, val in enumerate(assigned_binary) if val]

        self.label_box.value = tuple(assigned_names)

        with self.out:
            print(f"\nRow {self.cursor+1} / {len(self.df)} | Chunk ID: {row['filename']} ({row['year']})")
            print(f"Document Type: {row['document_type']} | Chunk #{row['chunk_num']}")
            print(f"Current Topics: {assigned_names}")
            print(f"Cosine: {row['cosine']} | Weighted: {row.get('cosine_weighted', 'n/a')}")
            print("\nText:\n" + "-"*80 + f"\n{row['chunk_text']}\n" + "-"*80)

    def approve_labels(self, _):
        selected = list(self.label_box.value)
        assigned_binary = [1 if topic in selected else 0 for topic in self.topic_names]

        self.df.at[self.cursor, 'topic_assigned'] = assigned_binary
        self.df.at[self.cursor, 'cosine'] = self.cosine_slider.value
        self.df.at[self.cursor, 'cosine_weighted'] = [self.cosine_slider.value if val else 0.0 for val in assigned_binary]
        self.next_row(None)

    def remove_labels(self, _):
        self.df.at[self.cursor, 'topic_assigned'] = [0] * self.num_topics
        self.df.at[self.cursor, 'cosine'] = 0.0
        self.df.at[self.cursor, 'cosine_weighted'] = [0.0] * self.num_topics
        self.next_row(None)

    def remove_row(self, _):
        self.df = self.df.drop(self.df.index[self.cursor]).reset_index(drop=True)
        self.cursor = max(0, min(self.cursor, len(self.df) - 1))
        self.show_current_row()

    def next_row(self, _):
        self.cursor = (self.cursor + 1) % len(self.df)
        self.show_current_row()

    def prev_row(self, _):
        self.cursor = (self.cursor - 1) % len(self.df)
        self.show_current_row()

    def save_to_csv(self, _):
        self.df['topic'] = self.df['topic'].apply(str)
        self.df['topic_assigned'] = self.df['topic_assigned'].apply(str)
        self.df.to_csv("relabelled_output.csv", index=False)
        print("✅ Saved to relabelled_output.csv")

# To use:
# relabeler = ChunkRelabeler("chunk_topics_corex_assignments.csv")


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# === Load and Prepare Data ===
file_path = "Corex_topicdata_slavery_policyv1\chunk_topics_corex_assignments.csv"
df = pd.read_csv(file_path)

def safe_parse_list(x):
    if isinstance(x, str):
        if x.startswith('['):
            try:
                return eval(x)
            except:
                return []
        elif '|' in x:
            return [t.strip() for t in x.split('|') if t.strip()]
        elif x.strip():
            return [x.strip()]
    return []

df['topic'] = df['topic'].apply(safe_parse_list)
df['topic_assigned'] = df['topic_assigned'].apply(safe_parse_list)
for col in ['cosine', 'cosine_weighted']:
    df[col] = df[col].apply(safe_parse_list)

all_topics = sorted(set(t for row in df['topic'] if isinstance(row, list) for t in row))

# === Interactive Labeling Tool ===
class TopicLabeler:
    def __init__(self, df, topic_names, sort_by='cosine_weighted'):
        self.original_df = df.copy()
        self.topic_names = sorted(set(topic_names))
        self.sort_by = sort_by
        self.df = self.sort_dataframe(df)
        self.index = 0
        self.build_ui()

    def sort_dataframe(self, df):
        if isinstance(df[self.sort_by].iloc[0], list):
            df = df.copy()
            df['sort_score'] = df[self.sort_by].apply(lambda x: max(x) if x else 0)
            return df.sort_values(by='sort_score', ascending=False).reset_index(drop=True)
        return df.sort_values(by=self.sort_by, ascending=False).reset_index(drop=True)

    def build_ui(self):
        self.out = widgets.Output(layout={'height': '300px', 'overflow': 'auto'})
        self.label_box = widgets.SelectMultiple(
            options=self.topic_names,
            description="Labels",
            layout=widgets.Layout(width='auto', height='140px')
        )
        self.cosine_slider = widgets.FloatSlider(
            value=1.0, min=0.0, max=1.0, step=0.01,
            description='Manual Cosine:'
        )
        self.sort_dropdown = widgets.Dropdown(
            options=['cosine', 'cosine_weighted'],
            description='Sort by:',
            value=self.sort_by
        )
        self.sort_dropdown.observe(self.change_sort, names='value')

        self.approve_button = widgets.Button(description="✅ Add / Update", button_style='success')
        self.remove_selected_button = widgets.Button(description="➖ Remove Selected", button_style='warning')
        self.clear_all_button = widgets.Button(description="❌ Clear All Labels", button_style='danger')
        self.remove_row_button = widgets.Button(description="🗑️ Remove Row", button_style='danger')
        self.skip_button = widgets.Button(description="⏭️ Skip", button_style='')
        self.next_button = widgets.Button(description="➡️ Next", button_style='info')
        self.prev_button = widgets.Button(description="⬅️ Prev", button_style='info')
        self.save_button = widgets.Button(description="💾 Save to CSV", button_style='primary')

        self.approve_button.on_click(self.approve_labels)
        self.remove_selected_button.on_click(self.remove_selected_labels)
        self.clear_all_button.on_click(self.remove_labels)
        self.remove_row_button.on_click(self.remove_row)
        self.next_button.on_click(self.next_row)
        self.prev_button.on_click(self.prev_row)
        self.save_button.on_click(self.save_to_csv)
        self.skip_button.on_click(self.skip_row)

        control_box = widgets.VBox([
            self.sort_dropdown,
            widgets.HBox([self.prev_button, self.next_button, self.skip_button, self.save_button]),
            self.label_box,
            self.cosine_slider,
            widgets.HBox([self.approve_button, self.remove_selected_button, self.clear_all_button, self.remove_row_button]),
            self.out
        ])
        display(control_box)
        self.show_current_row()

    def show_current_row(self):
        self.out.clear_output(wait=True)
        if 0 <= self.index < len(self.df):
            row = self.df.loc[self.index]
            with self.out:
                print(f"Chunk {self.index+1}/{len(self.df)}")
                print(f"Filename: {row['filename']} ({row['year']}) - {row['document_type']}")
                print(f"\nText:\n{'-'*80}\n{row['chunk_text']}\n{'-'*80}")
                print(f"\nCurrent Labels: {row['topic']}")
                print(f"Cosines: {row['cosine']}")
                print(f"Weighted Cosines: {row['cosine_weighted']}")

    def approve_labels(self, b):
        labels_to_add = list(self.label_box.value)
        cosine = self.cosine_slider.value

        current_labels = self.df.at[self.index, 'topic']
        current_cos = self.df.at[self.index, 'cosine']
        current_weighted = self.df.at[self.index, 'cosine_weighted']

        for label in labels_to_add:
            if label in current_labels:
                i = current_labels.index(label)
                current_cos[i] = cosine
                current_weighted[i] = cosine
            else:
                current_labels.append(label)
                current_cos.append(cosine)
                current_weighted.append(cosine)

        self.df.at[self.index, 'topic'] = current_labels
        self.df.at[self.index, 'cosine'] = current_cos
        self.df.at[self.index, 'cosine_weighted'] = current_weighted
        self.show_current_row()

    def remove_selected_labels(self, b):
        labels_to_remove = list(self.label_box.value)
        current_labels = self.df.at[self.index, 'topic']
        current_cos = self.df.at[self.index, 'cosine']
        current_weighted = self.df.at[self.index, 'cosine_weighted']

        # Loop backwards to prevent shifting issues, handle missing/extra labels safely
        indices = [i for i, t in enumerate(current_labels) if t in labels_to_remove]
        for i in sorted(indices, reverse=True):
            if i < len(current_labels):
                current_labels.pop(i)
            if i < len(current_cos):
                current_cos.pop(i)
            if i < len(current_weighted):
                current_weighted.pop(i)

        self.df.at[self.index, 'topic'] = current_labels
        self.df.at[self.index, 'cosine'] = current_cos
        self.df.at[self.index, 'cosine_weighted'] = current_weighted
        self.show_current_row()

    def remove_labels(self, b):
        self.df.at[self.index, 'topic'] = []
        self.df.at[self.index, 'cosine'] = []
        self.df.at[self.index, 'cosine_weighted'] = []
        self.show_current_row()

    def remove_row(self, b):
        self.df = self.df.drop(self.index).reset_index(drop=True)
        if self.index >= len(self.df):
            self.index = len(self.df) - 1
        self.show_current_row()

    def skip_row(self, b):
        self.index += 1
        if self.index >= len(self.df):
            self.index = len(self.df) - 1
        self.show_current_row()

    def next_row(self, b):
        if self.index < len(self.df) - 1:
            self.index += 1
            self.show_current_row()

    def prev_row(self, b):
        if self.index > 0:
            self.index -= 1
            self.show_current_row()

    def save_to_csv(self, b):
        self.df.to_csv("relabeled_output.csv", index=False)
        with self.out:
            print("✅ Data saved to relabeled_output.csv")

    def change_sort(self, change):
        self.sort_by = change['new']
        self.df = self.sort_dataframe(self.original_df)
        self.index = 0
        self.show_current_row()

# === Launch the Tool ===
labeler = TopicLabeler(df, all_topics)


: 